In [ ]:
"""
llm_screen.py
=============
Second-pass relevance screening with 3 LLMs (OpenAI, Gemini, DeepSeek), used to
adjudicate the rule-based tiers from tick_screen.py. All three providers speak
the OpenAI Chat Completions format, so there is ONE code path — only base_url,
api key, and model name differ.

What it produces (audit-friendly, one row per record)
-----------------------------------------------------
  keep for reference : oa_id, oa_title, oa_relevance_tier, oa_relevance_reason
  per model (gpt/gemini/deepseek):
      <m>_label       TICK_RELEVANT | NOT_TICK | UNCERTAIN | PARSE_ERROR
      <m>_conf        0-1 confidence the model reported
      <m>_reason      one-line justification (<=~30 words)
  consensus:
      llm_label       majority vote (ties -> UNCERTAIN)
      llm_agree       True if all models gave the same label
      llm_n_relevant  how many of the models voted TICK_RELEVANT
      match_vs_rule   agree | llm_stricter | llm_looser | resolved  <-- audit here

WHICH RECORDS TO SEND — you choose via `tiers`:
    tiers=["screen"]                      # default: adjudicate the ambiguous middle
    tiers=["screen","irrelevant"]         # also re-check the drops for false negatives
    tiers="all"                           # everything (expensive x3 models)
Plus `sample_per_tier` to validate a random sample of relevant/irrelevant cheaply.

Colab setup
-----------
    !pip -q install openai pandas pyarrow
    from google.colab import userdata
    import os
    os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
    os.environ["GEMINI_API_KEY"]   = userdata.get("GEMINI_API_KEY")
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")

    from llm_screen import screen, MODELS
    out = screen(df, tiers=["screen"], outdir="/content/drive/.../llm_screen")
    print(out["match_vs_rule"].value_counts())
"""

from __future__ import annotations
import os, re, json, time, glob, math
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

pd.set_option('display.max_columns', None)


# ---- provider config: all OpenAI-compatible; verify model IDs on pricing pages ----
MODELS = {
    "gpt":      dict(model="gpt-4o-mini",       base_url=None,                                            key_env="OPENAI_API_KEY"),
    "gemini":   dict(model="gemini-2.5-flash",  base_url="https://generativelanguage.googleapis.com/v1beta/openai/", key_env="GEMINI_API_KEY"),
    "deepseek": dict(model="deepseek-v4-flash", base_url="https://api.deepseek.com",                       key_env="DEEPSEEK_API_KEY"),
}

LABELS = ["TICK_RELEVANT", "NOT_TICK", "UNCERTAIN"]

SYSTEM = (
    "You are screening scientific records for a systematic review about TICKS "
    "(Ixodida: hard and soft ticks) and tick-borne diseases/pathogens, hosts, "
    "ecology, control, and vectors. Decide from ALL the provided metadata "
    "(title, abstract, keywords, MeSH terms, concepts, journal, topic) whether the "
    "record is genuinely about ticks or tick-borne disease.\n"
    "IMPORTANT: mosquitoes, fleas, lice, "
    "helminths, and the word 'tick' meaning a checkmark / clock tick / finance "
    "'tick size' are NOT relevant.\n"
    'Respond with ONLY a JSON object: {"label": one of '
    '["TICK_RELEVANT","NOT_TICK","UNCERTAIN"], "confidence": 0.0-1.0, '
    '"reason": "<=30 words"}. Use UNCERTAIN only if the metadata is genuinely '
    "insufficient to decide."
)

# Columns shown to the LLM, in order: (column, LABEL, max_chars). Edit freely.
# NOTE: oa_topic/oa_subfield/oa_field are OpenAlex's OWN classification — the
# rule-based tier already used them, so including them makes the LLM a less
# independent check. Left commented for that reason; uncomment for more context.
PROMPT_FIELDS = [
    ("oa_title",    "TITLE",       300),
    ("oa_abstract", "ABSTRACT",    2800),
    ("oa_keywords", "KEYWORDS",    500),
    ("oa_mesh",     "MeSH TERMS",  400),
    ("oa_concepts", "CONCEPTS",    300),
    ("oa_journal",  "JOURNAL",     120),
    ("oa_topic",    "OPENALEX TOPIC", 120),
    ("oa_subfield", "SUBFIELD",       80),
    ("oa_field",    "FIELD",          80),
]

_NA = {"", "nan", "none", "null", "[]", "{}", "n/a"}


def make_messages(row, fields=PROMPT_FIELDS) -> list[dict]:
    parts = []
    for col, label, n in fields:
        val = str(row.get(col) or "").strip()
        if val.lower() in _NA:
            continue
        val = val.replace(" | ", ", ")[:n]          # pipe-lists -> readable commas
        parts.append(f"{label}: {val}")
    user = "\n".join(parts) if parts else "TITLE: (no usable metadata)"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


# ---- clients (built once) ---------------------------------------------------
def build_clients(models=MODELS) -> dict:
    from openai import OpenAI
    clients = {}
    for name, cfg in models.items():
        key = os.environ.get(cfg["key_env"])
        if not key:
            print(f"  (skip {name}: {cfg['key_env']} not set)")
            continue
        clients[name] = OpenAI(api_key=key, base_url=cfg["base_url"])
    return clients


def _parse(txt: str) -> dict:
    t = (txt or "").strip()
    t = re.sub(r"^```(?:json)?|```$", "", t, flags=re.MULTILINE).strip()
    m = re.search(r"\{.*\}", t, re.DOTALL)
    if m:
        t = m.group(0)
    try:
        d = json.loads(t)
    except Exception:
        return {"label": "PARSE_ERROR", "conf": None, "reason": (txt or "")[:200]}
    label = str(d.get("label", "")).upper().strip().replace(" ", "_")
    if label not in LABELS:
        label = "UNCERTAIN" if "UNCERT" in label else ("NOT_TICK" if "NOT" in label else
                ("TICK_RELEVANT" if "RELEV" in label or "TICK" in label else "PARSE_ERROR"))
    conf = d.get("confidence")
    try:
        conf = round(float(conf), 2)
    except Exception:
        conf = None
    return {"label": label, "conf": conf, "reason": str(d.get("reason", ""))[:300]}


def _call(client, model, messages, retries=4):
    for a in range(retries):
        try:
            kw = dict(model=model, messages=messages, temperature=0, max_tokens=300)
            try:
                r = client.chat.completions.create(response_format={"type": "json_object"}, **kw)
            except Exception:
                r = client.chat.completions.create(**kw)   # provider rejected response_format
            return _parse(r.choices[0].message.content)
        except Exception as e:
            if a == retries - 1:
                return {"label": "API_ERROR", "conf": None, "reason": str(e)[:150]}
            time.sleep(2 ** a)


def _verdict_row(row, clients, fields=PROMPT_FIELDS) -> dict:
    msgs = make_messages(row, fields)
    out = {"oa_id": row.get("oa_id")}
    for name, client in clients.items():
        v = _call(client, MODELS[name]["model"], msgs)
        out[f"{name}_label"] = v["label"]
        out[f"{name}_conf"]  = v["conf"]
        out[f"{name}_reason"] = v["reason"]
    return out


# ---- consensus + audit ------------------------------------------------------
def _add_consensus(res: pd.DataFrame, model_names) -> pd.DataFrame:
    labs = [f"{m}_label" for m in model_names if f"{m}_label" in res.columns]

    def vote(r):
        vals = [r[c] for c in labs if r[c] in LABELS]
        if not vals:
            return "UNCERTAIN"
        from collections import Counter
        (lab, n), = Counter(vals).most_common(1)
        # tie between top-2 -> UNCERTAIN
        c = Counter(vals)
        top = c.most_common()
        if len(top) > 1 and top[0][1] == top[1][1]:
            return "UNCERTAIN"
        return lab

    res["llm_label"] = res.apply(vote, axis=1)
    res["llm_agree"] = res[labs].apply(lambda r: len(set(r.dropna())) == 1, axis=1)
    res["llm_n_relevant"] = (res[labs] == "TICK_RELEVANT").sum(axis=1)
    return res


def _match_vs_rule(tier: str, llm_label: str) -> str:
    rel = llm_label == "TICK_RELEVANT"
    not_rel = llm_label == "NOT_TICK"
    if tier == "relevant":
        return "agree" if rel else ("llm_stricter" if not_rel else "uncertain")
    if tier == "irrelevant":
        return "agree" if not_rel else ("llm_looser" if rel else "uncertain")
    if tier == "screen":
        return "resolved_relevant" if rel else ("resolved_not" if not_rel else "still_uncertain")
    return "n/a"


# ---- main -------------------------------------------------------------------
def screen(df: pd.DataFrame, tiers=("screen",), outdir="./llm_screen",
           sample_per_tier: int | None = None, models=MODELS, fields=PROMPT_FIELDS,
           workers: int = 8, flush_every: int = 200, max_records: int | None = None,
           seed: int = 0) -> pd.DataFrame:
    """Send the chosen tiers to the LLMs, checkpoint, and return df augmented
    with per-model + consensus + audit columns (only the screened rows carry them)."""
    out = Path(outdir); out.mkdir(parents=True, exist_ok=True)

    # 1) select records
    if tiers == "all":
        target = df.copy()
    else:
        target = df[df["oa_relevance_tier"].isin(list(tiers))].copy()
        if sample_per_tier:
            extra = (df[~df["oa_relevance_tier"].isin(list(tiers))]
                     .groupby("oa_relevance_tier", group_keys=False)
                     .apply(lambda g: g.sample(min(sample_per_tier, len(g)), random_state=seed)))
            target = pd.concat([target, extra])
    if max_records:
        target = target.head(max_records)
    print(f"Records to screen: {len(target):,}")

    # 2) resume: skip oa_ids already checkpointed
    done_ids = set()
    for p in sorted(glob.glob(str(out / "part_*.parquet"))):
        done_ids |= set(pd.read_parquet(p, columns=["oa_id"])["oa_id"])
    todo = target[~target["oa_id"].isin(done_ids)]
    print(f"  already done: {len(done_ids):,} | remaining: {len(todo):,}")

    clients = build_clients(models)
    if not clients:
        raise RuntimeError("No API keys set — nothing to call.")

    # 3) run with a thread pool, checkpointing in flushes
    part_no = len(glob.glob(str(out / "part_*.parquet")))
    rows = todo.to_dict("records")
    buf = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(_verdict_row, r, clients, fields): i for i, r in enumerate(rows)}
        for k, fut in enumerate(as_completed(futs), 1):
            buf.append(fut.result())
            if len(buf) >= flush_every:
                pd.DataFrame(buf).to_parquet(out / f"part_{part_no:05d}.parquet", index=False)
                part_no += 1; buf = []
                print(f"    ...{k:,}/{len(rows):,} judged")
    if buf:
        pd.DataFrame(buf).to_parquet(out / f"part_{part_no:05d}.parquet", index=False)

    # 4) combine verdicts, add consensus, merge onto df
    parts = sorted(glob.glob(str(out / "part_*.parquet")))
    res = pd.concat((pd.read_parquet(p) for p in parts), ignore_index=True).drop_duplicates("oa_id")
    res = _add_consensus(res, list(clients.keys()))

    merged = df.merge(res, on="oa_id", how="left")
    m = merged["llm_label"].notna()
    merged.loc[m, "match_vs_rule"] = [
        _match_vs_rule(t, l) for t, l in
        zip(merged.loc[m, "oa_relevance_tier"], merged.loc[m, "llm_label"])
    ]
    merged.to_parquet(out / "screened_llm.parquet", index=False)
    # slim audit CSV
    audit_cols = (["oa_id", "oa_title", "oa_relevance_tier", "oa_relevance_reason",
                   "llm_label", "llm_agree", "llm_n_relevant", "match_vs_rule"]
                  + [c for mdl in clients for c in (f"{mdl}_label", f"{mdl}_conf", f"{mdl}_reason")])
    audit_cols = [c for c in audit_cols if c in merged.columns]
    merged.loc[m, audit_cols].to_csv(out / "audit.csv", index=False)
    print(f"Saved {out/'screened_llm.parquet'} and {out/'audit.csv'}")
    return merged


def report(df: pd.DataFrame) -> str:
    lines = ["LLM SCREEN — consensus vs rules"]
    if "match_vs_rule" in df.columns:
        for k, v in df["match_vs_rule"].value_counts().items():
            lines.append(f"  {k:20s}: {v:>7,}")
    if "llm_agree" in df.columns:
        m = df["llm_label"].notna()
        lines.append(f"  3-model full agreement: {df.loc[m,'llm_agree'].mean():.1%}")
    return "\n".join(lines)

In [ ]:

# ---- 1. mount + locate the file ----
from google.colab import drive
drive.mount('/content/drive')

FILENAME = "canonical_screened.parquet"          # or "combined.parquet"
hits = glob.glob(f"/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/{FILENAME}", recursive=True)
assert hits, (f"{FILENAME} not found under /content/drive. Is Drive synced? "
              f"Open the Files panel and confirm /content/drive is populated.")
PATH   = hits[0]
OUTDIR = os.path.join(os.path.dirname(PATH), "llm_screen")
print("Using :", PATH)
print("Output:", OUTDIR)

In [ ]:
!pip -q install openai pandas pyarrow
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
os.environ["GEMINI_API_KEY"]   = userdata.get("GEMINI_API_KEY")
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
print(build_clients().keys())

# from llm_screen import screen, MODELS
df = pd.read_parquet(PATH)
out = screen(df, tiers=["relevant"], outdir=OUTDIR) # , "relevant" screen
print(report(out))

In [ ]:
out[out.llm_label.notna()].to_csv("/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/llm_screen_test.csv", index=False)

In [ ]:
out

In [ ]:
!pip -q install openai pandas pyarrow
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
os.environ["GEMINI_API_KEY"]   = userdata.get("GEMINI_API_KEY")
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
print(build_clients().keys())

# from llm_screen import screen, MODELS
df = pd.read_parquet(PATH)
out = screen(df, tiers=["screen"], outdir=OUTDIR) # , "relevant"
print(report(out))

In [ ]:
!pip -q install openai pandas pyarrow
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
os.environ["GEMINI_API_KEY"]   = userdata.get("GEMINI_API_KEY")
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
print(build_clients().keys())

# from llm_screen import screen, MODELS
df = pd.read_parquet(PATH)
out = screen(df, tiers=["screen"], outdir=OUTDIR) # , "relevant"
print(report(out))

In [ ]:
ccc

In [ ]:
!pip -q install openai pandas pyarrow
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
# os.environ["GEMINI_API_KEY"]   = userdata.get("GEMINI_API_KEY")
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
print(build_clients().keys())

# from llm_screen import screen, MODELS
df = pd.read_parquet(PATH)
out = screen(df, tiers=["screen", "relevant", "irrelevant"], max_records=20, outdir=OUTDIR)
print(report(out))

In [ ]:
out[out.llm_label.notna()]

In [ ]:
out